# Week 3, day 3 (afternoon) — Worksheet 05 SOLUTIONS: validating the load

Executed in the lab image against the real lab CSVs; every quoted figure was
observed.

Read Q9 and Q10 together. Q9 is a defect in the data. Q10 is a defect in the
script that is supposed to catch defects — and it is the more expensive of the
two.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 05 — Validating the load. Run this once.
import pandas as pd

STG_SALES_COLS = [
    "TRANS_ID", "PROD_KEY", "STORE_KEY", "TRANS_DT", "TRANS_TIME",
    "PRIORITY", "SALES_QTY", "SALES_PRICE", "SALES_AMT",
    "DISCOUNT", "SALES_COST", "SALES_MGRN", "SHIP_MODE", "SHIP_COST",
]
STG_PRODUCT_COLS = [
    "PROD_KEY", "PROD_NAME", "VOL", "WGT", "BRAND_NAME",
    "STATUS_CODE", "STATUS_CODE_NAME", "CATEGORY_KEY", "CATEGORY_NAME",
    "SUBCATEGORY_KEY", "SUBCATEGORY_NAME",
]


def load_stg_sales():
    """STG_Sales exactly as 3_Stage_tables.sql builds it, quotes stripped."""
    df = pd.read_csv("data/sales_2013_01_01.csv", header=None, skiprows=1,
                     names=STG_SALES_COLS)
    for col in ("PRIORITY", "SHIP_MODE"):
        df[col] = df[col].str.strip('"')
    df["TRANS_DT"] = pd.to_datetime(df["TRANS_DT"], format="%m/%d/%Y")
    df["BATCH_ID"] = "sales_2013_01_01.csv"
    return df


def load_stg_products():
    df = pd.read_csv("data/products_2013_01_01.csv", header=None, skiprows=1,
                     names=STG_PRODUCT_COLS)
    df["BATCH_ID"] = "products_2013_01_01.csv"
    return df


sales = load_stg_sales()
products = load_stg_products()
print("STG_Sales:   ", sales.shape)
print("STG_Products:", products.shape)

PART A — the check the lab runs

### Question 1

Reproduce `4_Validate_stage_tables.sql` for both tables: group by `BATCH_ID` and count rows. Print both results.

In [ ]:
print("STG_Products:")
print(products.groupby("BATCH_ID").size().rename("row_count").to_string())
print()
print("STG_Sales:")
print(sales.groupby("BATCH_ID").size().rename("row_count").to_string())

```
STG_Products:
BATCH_ID
products_2013_01_01.csv    1215

STG_Sales:
BATCH_ID
sales_2013_01_01.csv    100000
```

One batch each, 1,215 and 100,000 rows. This is a pass. The lab is finished, the
data is in, and everything below happens *after* this result.

### Question 2

Show what that check is for. Simulate re-running the COPY by concatenating `sales` with a second `load_stg_sales()`, then run the same group-and-count. Print the count before and after.
> **NOTE:** `COPY INTO` skips files it has already loaded — for 64 days, tracked per table. `FORCE = TRUE` or a stage reset defeats that.

In [ ]:
again = pd.concat([sales, load_stg_sales()], ignore_index=True)
print("after one COPY: ", len(sales))
print("after two COPYs:", len(again))
print()
print(again.groupby("BATCH_ID").size().rename("row_count").to_string())
print()
print("distinct TRANS_ID is unchanged:", sales["TRANS_ID"].nunique(),
      "->", again["TRANS_ID"].nunique())

```
after one COPY:  100000
after two COPYs: 200000

BATCH_ID
sales_2013_01_01.csv    200000

distinct TRANS_ID is unchanged: 7916 -> 7916
```

This is what the check earns you. A doubled load shows up immediately as
200,000, and 200,000 is obviously wrong for a file with 100,000 lines. Worth
having, and worth running.

Two details make it less reassuring than it looks.

`BATCH_ID` is still one value. The second load is indistinguishable from the
first inside the table — same filename, same tag — so `BATCH_ID` tells you the
rows came from that file but not that they came from it twice. Only the count
does. (`INSERTED_AT` would separate them, which is why the lab's query groups by
it as well.)

And `COUNT(DISTINCT TRANS_ID)` did not move: 7,916 before, 7,916 after. Any
report built on distinct counts would have shown nothing at all, while every
`SUM` on the table silently doubled.

In practice Snowflake usually prevents this for you — `COPY INTO` records the
files it has loaded per table and skips them for 64 days. That is real protection
and it is also why `FORCE = TRUE`, a recreated stage, or a re-uploaded file with
a new path each defeat it without warning.

PART B — the checks it does not run

### Question 3

Write a small completeness report over `sales`: for every column, the number of nulls and the number of distinct values. Print it sorted by distinct count.

In [ ]:
report = pd.DataFrame({
    "nulls": sales.isna().sum(),
    "distinct": sales.nunique(),
})
print(report.sort_values("distinct").to_string())

```
             nulls  distinct
BATCH_ID         0         1
SHIP_MODE        0         3
PRIORITY         0         5
STORE_KEY        0        13
TRANS_TIME       0        14
DISCOUNT         0        16
SALES_QTY        0       166
SHIP_COST        0       617
SALES_PRICE      0       720
PROD_KEY         0      1212
TRANS_DT         0      1411
SALES_MGRN       0      7398
SALES_COST       0      7715
TRANS_ID         0      7916
SALES_AMT        0     27009
```

**Zero nulls in every column.** By the usual data-quality checklist this table is
spotless, which is the reason to have a second checklist.

The distinct counts are where the information is, and they sort into three
groups. Under about 20 — `SHIP_MODE`, `PRIORITY`, `STORE_KEY`, `TRANS_TIME`,
`DISCOUNT` — these are categories, and you can eyeball every value they contain
(Q4 does). Around a thousand — `PROD_KEY` at 1,212, `TRANS_DT` at 1,411 — these
are dimensions, and the right check is whether the set is complete (Q5). In the
tens of thousands — the money columns — these are measures, and only their
distribution can be checked (Q7).

Two anomalies are already visible. `TRANS_TIME` has 14 distinct values, so it is
an hour bucket, not a time. And `TRANS_ID` has fewer distinct values (7,916) than
`SALES_AMT` (27,009), which is only possible because `TRANS_ID` repeats — as
Worksheet 03 found.

This one small table is the cheapest orientation you can buy on an unfamiliar
dataset. Run it first, before writing any query.

### Question 4

`STORE_KEY` has very few distinct values. Print the row count per store, sorted descending, and the share of total rows held by the largest one.

In [ ]:
per_store = sales["STORE_KEY"].value_counts()
print(per_store.to_string())
print()
print("stores:", len(per_store))
print("largest share: %.1f%%" % (100 * per_store.iloc[0] / len(sales)))

```
STORE_KEY
8107    7916
9001    7916
...
9012    7916
8106    7140
9013    5784

stores: 13
largest share: 7.9%
```

Thirteen stores, and eleven of them have **exactly 7,916 rows** — which is
exactly the number of distinct `TRANS_ID` values.

That is the grain confirmed from the other direction. Each `TRANS_ID` is one
product on one day, and it appears once per store, so a store that stocked every
product-day has one row per transaction: 7,916. Store 8106 has 7,140 and store
9013 has 5,784, meaning they were absent from some product-days — a store that
opened late, closed early, or does not carry the full range.

Identical counts across a dimension are a fingerprint of generated or fully
cross-joined data, and they are worth noticing before you present "sales by
store" as a business finding. Eleven stores tied to the row is not what real
retail looks like; the variation lives in the amounts, not the row counts.

Store 9013's 5,784 is the interesting number — 27% below the others. Whether that
is a genuine gap or a partial extract is a question for whoever produced the
file, and it is the kind of question you can only ask if you looked.

### Question 5

Check the date column for gaps. Build a full daily range from the minimum to the maximum `TRANS_DT` and print the total days in the range, the days present, and the days missing.

In [ ]:
days = sales["TRANS_DT"].dt.normalize()
full = pd.date_range(days.min(), days.max(), freq="D")
present = pd.Index(days.unique()).sort_values()
missing = full.difference(present)
print("range:        ", full.min().date(), "to", full.max().date())
print("days in range:", len(full))
print("days present: ", len(present))
print("days missing: ", len(missing))
print()
print("first few missing:", [str(d.date()) for d in missing[:5]])

```
range:         2009-01-01 to 2012-12-30
days in range: 1460
days present:  1411
days missing:  49
```

Forty-nine days in the range have no rows at all. Not nulls — absences, which no
null check can see, because there is nothing there to be null.

This is the check that catches the failure mode row counts are worst at: a load
that is complete in the sense that every line of the file arrived, and incomplete
in the sense that the file never had those days in it. 1,411 of 1,460 is 96.6%
coverage, which sounds fine and is not the same as "four years of daily sales".

Note that the endpoints are also information: the range stops at **2012-12-30**,
not the 31st, and starts on 2009-01-01. A file named for 2013-01-01 containing
nothing after 30 December 2012 is worth a question on its own.

Q6 asks the obvious follow-up.

### Question 6

Look at the missing days. Print how many of them fall on each weekday name, using `missing.day_name()`.

In [ ]:
days = sales["TRANS_DT"].dt.normalize()
full = pd.date_range(days.min(), days.max(), freq="D")
missing = full.difference(pd.Index(days.unique()))
print(pd.Series(missing.day_name()).value_counts().to_string())
print()
present = pd.Index(days.unique())
print("Sundays in range:  ", int((full.day_name() == "Sunday").sum()))
print("Sundays with sales:", int((present.day_name() == "Sunday").sum()))

```
Saturday     9
Wednesday    9
Tuesday      7
Thursday     7
Sunday       6
Friday       6
Monday       5

Sundays in range:   209
Sundays with sales: 203
```

The gaps are spread across all seven weekdays — 9, 9, 7, 7, 6, 6, 5 — with no
weekday missing more than a couple more than any other. And 203 of the 209
Sundays in the range **do** have sales.

So the answer is no: this is not weekend or holiday closure. Had it been, the 49
missing days would have been 49 Sundays, or clustered on public holidays, and
there would be a business explanation requiring no action.

There isn't one. The gaps look random, which points at the extract rather than
the business — a job that failed on 49 days over four years and was never
backfilled, or a source that drops days silently.

The question is worth asking in this order, and this is the general point of Q5
and Q6 together. Finding a gap is not a finding. A gap with an explanation is
normal; a gap without one is a bug, and you cannot tell which you have until you
test the obvious explanation and watch it fail.

PART C — numbers that pass every null check

### Question 7

Print `describe()` for `SALES_QTY`, `SALES_AMT`, `SALES_COST` and `SALES_MGRN`, rounded to 2dp. Note the minimum of each.

In [ ]:
cols = ["SALES_QTY", "SALES_AMT", "SALES_COST", "SALES_MGRN"]
print(sales[cols].describe().round(2).to_string())

```
       SALES_QTY  SALES_AMT  SALES_COST  SALES_MGRN
count  100000.00  100000.00   100000.00   100000.00
mean       21.72    1465.62     1599.27      191.21
std        12.72    3033.34     3093.26     1214.43
min         0.70       1.57       -8.31   -14140.70
25%        10.80     120.27      161.23      -79.81
50%        21.60     369.32      460.93       -0.06
75%        32.00    1401.75     1627.01      168.54
max        50.00   89061.05    60958.57    27220.69
```

Three things in the `min` row and one in the middle.

**`SALES_COST` has a minimum of -8.31.** A negative cost. Whatever that is — a
refund, a sign error, a credit note posted to the wrong column — it is not a cost,
and `SUM(SALES_COST)` is quietly absorbing it.

**`SALES_MGRN` has a minimum of -14,140.70**, and its 25th percentile is -79.81,
so at least a quarter of all rows lose money. Q8 counts them properly.

**The median margin is -0.06** — essentially zero, just below it. Half the rows
are at or under break-even.

And `mean` 1,465.62 against `median` 369.32 for `SALES_AMT`, with a max of
89,061.05: a long right tail, so any "average sale" figure will be four times the
typical sale. Report the median.

None of this violates a constraint. Every value is non-null, numeric, and in
range for a `FLOAT`. `describe()` on the numeric columns costs one line and is
the only check here that would have surfaced it.

### Question 8

Count the rows where `SALES_MGRN` is negative, and their share of the total. Then check whether the margin is at least self-consistent: how many rows satisfy `SALES_MGRN == SALES_AMT - SALES_COST` to 2dp?

In [ ]:
neg = sales["SALES_MGRN"] < 0
print("rows with negative margin:", int(neg.sum()),
      "(%.1f%%)" % (100 * neg.sum() / len(sales)))
print()
calc = (sales["SALES_AMT"] - sales["SALES_COST"]).round(2)
agree = int((calc == sales["SALES_MGRN"].round(2)).sum())
print("rows where AMT - COST == MGRN:", agree, "of", len(sales))
print()
print("total SALES_AMT:  %.2f" % sales["SALES_AMT"].sum())
print("total SALES_COST: %.2f" % sales["SALES_COST"].sum())
print("total SALES_MGRN: %.2f" % sales["SALES_MGRN"].sum())

```
rows with negative margin: 50039 (50.0%)

rows where AMT - COST == MGRN: 41 of 100000

total SALES_AMT:  146561868.39
total SALES_COST: 159927036.83
total SALES_MGRN: 19121304.17
```

**Exactly half the rows lose money** — 50,039 of 100,000, which is 50.0%. For real
retail that is not a data-quality flag, it is a going-out-of-business flag. For
generated data it is the giveaway: the margins were drawn from a distribution
centred on zero.

Then the consistency check, and this is the worse finding. `SALES_AMT - SALES_COST`
equals `SALES_MGRN` in **41 rows out of 100,000**. The three columns do not describe
the same transaction.

The totals show it does not wash out in aggregate either. Revenue 146.6M, cost
159.9M — so the sum of the parts is a **loss of 13.4M** — while the margin column
sums to a **profit of 19.1M**. Same table, same 100,000 rows, two answers 32
million apart, and both are just a `SUM` any analyst would write.

Together with Worksheet 03 Q9 (`SALES_AMT` equals `QTY x PRICE` in 33 rows), the
picture is that all six money columns were generated independently. There is no
arithmetic that reconciles them, so there is no cleaning step that fixes this.

Which makes it a decision, not a defect to repair: the model has to declare which
column is authoritative for revenue and which for margin, and that declaration
has to be written where the next person will find it. The alternative is two
dashboards that disagree and nobody able to say which is right.

PART D — what a join does with a duplicated key

### Question 9

Join `sales` to `products` on `PROD_KEY` with an inner merge. Print the row count before and after, and the total `SALES_AMT` before and after.
> **NOTE:** Worksheet 03 Q7 found three duplicate `PROD_KEY` rows in a 1,215-row table. This is what they cost.

In [ ]:
joined = sales.merge(products[["PROD_KEY", "CATEGORY_NAME", "BRAND_NAME"]],
                     on="PROD_KEY", how="inner")
print("sales rows:  ", len(sales))
print("joined rows: ", len(joined))
print("rows gained: ", len(joined) - len(sales))
print()
print("SALES_AMT before join: %.2f" % sales["SALES_AMT"].sum())
print("SALES_AMT after join:  %.2f" % joined["SALES_AMT"].sum())
print("inflated by:           %.2f (%.3f%%)"
      % (joined["SALES_AMT"].sum() - sales["SALES_AMT"].sum(),
         100 * (joined["SALES_AMT"].sum() / sales["SALES_AMT"].sum() - 1)))

```
sales rows:   100000
joined rows:  100324
rows gained:  324

SALES_AMT before join: 146561868.39
SALES_AMT after join:  147168574.31
inflated by:           606705.92 (0.414%)
```

Three duplicate rows in a 1,215-row dimension produced **324 extra fact rows** and
**606,705.92 of revenue that does not exist**.

The mechanism: two of the duplicated products (`72479` and `481924`, from
Worksheet 03 Q7) have contradictory `CATEGORY_NAME` values, so each sale of them
matches two product rows and comes out of the merge twice. The sale is counted
once under each category.

Look at the size of the error — 0.414%. That is the dangerous magnitude. A number
that is 50% wrong gets caught in review; a number that is 0.4% wrong looks like
rounding, sits inside every plausible tolerance, and gets signed off. It will also
move with the product mix, so the same report will be wrong by a different small
amount each month.

Note what the row count would have told you if you had checked it: 100,324 rows
out of a 100,000-row fact table means the join fanned out, and that check is one
comparison. It is the check to run after every join to a dimension you did not
build yourself.

The fix belongs upstream — deduplicate `STG_Products` on `PROD_KEY` with an
explicit rule before anything joins to it. Deduplicating the *result* of the join
does not work, because by then the duplicated rows are indistinguishable from
legitimate ones.

### Question 10

Finally, read STEP 3 of `4_Validate_stage_tables.sql` — `DROP TABLE CORE.DIM_CALENDAR;` — then search every file in `snowflake-scripts/` and `labs/` for `DIM_CALENDAR`. Print each hit with its file and line, and count how many of them CREATE the table. Then say what happens when a student runs all four scripts in order.
> **NOTE:** read the scripts; this question is not about pandas. `glob.glob("snowflake-scripts/*.sql") + glob.glob("labs/*.html")`.

In [ ]:
import glob
import os

paths = sorted(glob.glob("snowflake-scripts/*.sql")) + sorted(glob.glob("labs/*.html"))
print("searched %d files:" % len(paths))
for p in paths:
    print("   ", p)
print()

hits = []
for path in paths:
    for i, line in enumerate(open(path, errors="replace"), 1):
        if "DIM_CALENDAR" in line.upper():
            hits.append((os.path.basename(path), i, line.strip()))

for name, i, line in hits:
    print("%-30s line %2d: %s" % (name, i, line))
print("mentions of DIM_CALENDAR:", len(hits))
print("of those, statements that CREATE it:",
      sum(1 for _, _, l in hits if "CREATE" in l.upper()))

```
searched 6 files:
    snowflake-scripts/1_environment_setup.sql
    snowflake-scripts/2_create_internal_stage.sql
    snowflake-scripts/3_Stage_tables.sql
    snowflake-scripts/4_Validate_stage_tables.sql
    labs/SDA_Lab_Getting_Started_with_Snowflake_and_Snowsight.html
    labs/SDA_lab_Ingest_Data_SF.html

4_Validate_stage_tables.sql    line 38: DROP TABLE CORE.DIM_CALENDAR;
mentions of DIM_CALENDAR: 1
of those, statements that CREATE it: 0
```

One mention across all six files, and it is the `DROP`. Nothing in this lab ever
creates `CORE.DIM_CALENDAR`.

So a student who runs the four scripts in order gets an error on the last
statement — Snowflake's *"Table 'DIM_CALENDAR' does not exist or not
authorized"* — after everything else has worked. The load is fine. The lab is
finished. The last thing on the screen is a failure with no cause in anything
they did.

It is a leftover: the script was written against an environment where that table
existed, and the cleanup step outlived the thing it cleaned up. Harmless here,
because dropping a table that does not exist does nothing.

But look at what it is attached to. This is the file called
`4_Validate_stage_tables.sql` — the *validation* script, the one whose output you
would trust to tell you the load is sound. Its last statement is a `DROP TABLE`
against a different schema. Anyone reading the console after a full run has to
decide whether the red text means their ingestion failed, and the honest answer
is that they cannot tell from the script.

Two habits come out of this worksheet, and this question is the second one.

**Validate the data, not just the load.** Every finding above — 49 missing days,
50,039 negative margins, three money columns that disagree by 32 million, a join
that invents 606,705.92 — was sitting in a table that passed
`4_Validate_stage_tables.sql` cleanly. Row counts prove arrival. Nulls, distinct
values, ranges, gaps, duplicate keys and join fan-out prove rather more, and cost
about ten lines each.

**Then validate the validation.** A check nobody has run end to end is not a
check. This one has a statement in it that fails every time, in the file whose
whole job is to tell you whether something failed — and it shipped, because it
was never run in a clean environment. Read your own pipeline's output at least
once, from an empty database, before you rely on it to tell you the truth.